# PolicyProof AI — create a masked dispute-only adjudication packet (CPU)

Use this **only after two genuinely independent reviewers have completed all 96 responses** in the v2 blank review packet. This notebook never runs a model, never creates human votes, and never certifies any reviewer.

You must upload these **four distinct files with these filenames**: `dev48_blank.csv` (the original untouched v2 packet), `dev48_private_mapping.json` (owner-only), `reviewer_A.csv` and `reviewer_B.csv` (the two completed human assessments). Do **not** upload any of these to public GitHub. Use **Runtime → CPU**. The packet eventually given to a third reviewer will contain NO original first-reviewer votes, gold categories or model identity.

## 1 — Obtain latest public code and prepare synthetic frozen case fixture

In [ ]:
from pathlib import Path
import subprocess, sys

repo = Path("/content/PolicyProof-AI")
if not repo.exists():
    subprocess.run([
        "git", "clone", "--depth", "1",
        "https://github.com/betterlifestyle04-commits/PolicyProof-AI.git",
        str(repo),
    ], check=True)
assert (repo/"scripts/adjudicate_dev48_review.py").is_file(), (
    "This checkout is older than the masked adjudication support. "
    "Use a fresh Colab session or a fresh clone."
)
subprocess.run([sys.executable, "-m", "scripts.prepare"], cwd=repo, check=True)
print("CPU-only frozen dataset prepared; no model weights or GPU used.")

## 2 — Upload the four preserved private files (exclusively, without overwrite)

The two `reviewer_*.csv` files must contain **real**, complete, independent judgments. Never copy reviewer scores from another file or use a language model to fabricate human labels. Renaming an actual completed file is fine; altering its votes is not.

In [ ]:
from google.colab import files

names = {
    "dev48_blank.csv",
    "dev48_private_mapping.json",
    "reviewer_A.csv",
    "reviewer_B.csv",
}
uploaded = files.upload()
if set(uploaded) != names:
    raise ValueError("Upload EXACTLY the four named review files, no extras")
private_dir = repo / "reviews"
private_dir.mkdir(parents=True, exist_ok=True)
if private_dir.is_symlink():
    raise ValueError("Private reviews directory may not be a symlink")
for name in sorted(names):
    content = uploaded[name]
    if not 0 < len(content) < 4_000_000:
        raise ValueError("Review input unexpectedly empty or oversized: " + name)
    with (private_dir/name).open("xb") as handle:
        handle.write(content)
print("Private evidence restored without overwriting any existing file.")

## 3 — Validate independent reviewer files, and prepare a blank dispute packet only if needed

The original review CSVs are read and checked for full coverage, matching answers and a stable packet SHA. If they agree completely, there is no reason to contact a third adjudicator. If they disagree, this creates two NEW owner-only outputs: a reviewer-safe blank CSV and a confidential checksum manifest.

In [ ]:
import json
sys.path.insert(0, str(repo))
from scripts.paired_semantic_review import summarize_reviews
from scripts.adjudicate_dev48_review import export_disputes

review_names = ["reviews/reviewer_A.csv", "reviews/reviewer_B.csv"]
summary = summarize_reviews(
    repo, "reviews/dev48_blank.csv",
    "reviews/dev48_private_mapping.json", review_names,
)
print("Original reviewer status:", summary["status"])
print("Disputed blinded responses:", summary["disputed_response_rows"])
assert summary["reviewer_identities_or_competence_verified"] is False
if summary["disputed_response_rows"]:
    result = export_disputes(repo, reviewer_files=review_names)
    assert result["original_votes_shared_with_adjudicator"] is False
    assert result["human_adjudication_completed"] is False
    print("Separate masked dispute packet generated:", result["rows_needing_independent_adjudication"])
else:
    print("No disputed responses: do NOT create or request third-party votes.")

## 4 — Download only the appropriate files

If any disagreements exist, download and safely back up BOTH outputs. **Share ONLY `dev48_disputes_blank.csv` with an independent qualified adjudicator.** Do NOT share `dev48_disputes_private_manifest.json`, the original A/B key, first reviewer votes or derived scores.

The adjudicator should fill all three 0/1 verdict fields for each disputed row, use a distinct `reviewer_code`, leave frozen evidence columns unchanged, and return a file named `dev48_adjudicator_completed.csv`. Return later to the [separate resolution notebook](https://colab.research.google.com/github/betterlifestyle04-commits/PolicyProof-AI/blob/main/notebooks/10_dev48_blind_dispute_resolve_cpu.ipynb) with the seven original/derived private files.

In [ ]:
if summary["disputed_response_rows"]:
    files.download(str(repo/"reviews/dev48_disputes_blank.csv"))
    files.download(str(repo/"reviews/dev48_disputes_private_manifest.json"))
    print("Review-safe blank CSV: share with third human.")
    print("Private manifest: owner ONLY, do NOT share.")
else:
    print("No third-party adjudication packet required. Preserve both original reviews.")